In [1]:
import pandas as pd
import numpy as np
import nfl_data_py as nfl
import statsmodels.formula.api as smf
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
seasons = range(2022, 2024+1)
pbp = nfl.import_pbp_data(seasons)

2022 done.
2023 done.
2024 done.
Downcasting floats.


In [21]:
print(pbp.filter(like = 'personnel').columns)

Index(['offense_personnel', 'defense_personnel'], dtype='object')


In [4]:
# Calculate basic metrics for lions passing game.
# Find Yards per pass, air yards per pass, 

In [5]:
lions_pbp = pbp\
    .query('posteam == "DET" & season_type == "REG"')

In [6]:
# Start with Yards per pass rank.
# Create dataframe for all passing plays in regular season during 2022-2024

passing = pbp\
    .query('play_type == "pass" &  season_type == "REG"')

sack_plays = passing\
    .query('sack == 1')

In [7]:
# Figure out why the numbers on this are different from numbers on Pro Football Reference
# Sack yardage is included in passing yards for some reason.
# you have to calculate sack yardage and then subtract it from passing yards


# Get passing yards per completion
passing_ypc = passing\
    .groupby(['season', 'posteam'])\
    .agg({'passing_yards': ['mean', 'count', 'sum'],
           'pass_attempt': 'count'})    

passing_ypc.columns = list(map('_'.join, passing_ypc.columns.values))

passing_ypc.reset_index(inplace = True)

# get sack yardage
sack_yardage = sack_plays\
    .groupby(['season', 'posteam'])\
    .agg({'yards_gained': 'sum'})\

# merge the passing_ypc and sack_yardage dataframes. 
# you have to join on both season and posteam because you want each teams numbers by season
adjusted_passing_yards = pd.merge(passing_ypc, sack_yardage,
                                  on = ['season', 'posteam'],
                                  how = 'inner'
                                 )

# rename all the columns for cleanliness
adjusted_passing_yards.rename(columns = {'passing_yards_count': 'completions',
                                         'pass_attempt_count': 'attempts',
                                         'passing_yards_sum': 'total',
                                         'yards_gained': 'sack_yardage',
                                         'passing_yards_mean': 'ypc'},
                                         inplace = True)

# calculate adjusted passing yards and  adjusted_ypc 
# This is passing yards - sack yards
# adjusted_passing_yards['adjusted_total'] = adjusted_passing_yards['total'] + adjusted_passing_yards['sack_yardage']
# adjusted_passing_yards['adjusted_ypc'] = adjusted_passing_yards['adjusted_total'] / adjusted_passing_yards['completions']

#adjusted_passing_yards['ypc'] = adjusted_passing_yards['ypc'].round(2)
#adjusted_passing_yards['adjusted_ypc'] = adjusted_passing_yards['adjusted_ypc'].round(2)

# create a ranking system where highest yards per completion being rank 1
adjusted_passing_yards['rank'] = adjusted_passing_yards\
                                .groupby('season')['ypc']\
                                .rank(ascending = False, method = 'min')

print(adjusted_passing_yards\
      .query('posteam == "DET"')\
      .sort_values(['season', 'rank'], ascending = [False, True]))

    season posteam        ypc  completions   total  attempts  sack_yardage  \
74    2024     DET  11.824561          399  4718.0       582        -244.0   
42    2023     DET  11.289216          408  4606.0       638        -205.0   
10    2022     DET  11.603133          383  4444.0       612        -163.0   

    rank  
74   7.0  
42   9.0  
10  11.0  


In [8]:
# TENDECIES

In [9]:
# Run VS Pass Percentages
# Run vs Pass by down
# Pass location
# run location
# Formations
# Personnel by down

In [10]:
# Pun vs Pass Percentages

lions_pass = pbp\
    .query('play_type == "pass" & season_type == "REG" & posteam == "DET"')

lions_run = pbp\
    .query('play_type == "run" & season_type == "REG" & posteam == "DET"')

In [11]:
# Create pass count and run count dataframes. Calculate pass vs run percentages
# You can rename a column in the .agg function

pass_count = lions_pass\
    .groupby(['season','down', 'play_type'])\
    .agg(pass_count = ('play_type', 'count'))

pass_count.reset_index(inplace = True)

run_count = lions_run\
    .groupby(['season','down', 'play_type'])\
    .agg(run_count = ('play_type', 'count'))

run_count.reset_index(inplace = True)

scrambles = lions_pbp\
    .query('qb_scramble == 1')

scramble_count = scrambles\
    .groupby(['season','down'])\
    .agg({'qb_scramble': 'count'})


# First merge pass_count and run_count
play_count = pd.merge(pass_count, run_count, 
                      on=['season', 'down'], 
                      how='outer')\
                      .fillna(0)

# Then merge with scramble_count
play_count = pd.merge(play_count, scramble_count,
                      on=['season','down'], 
                      how='outer')\
                      .fillna(0)


play_count['pass_count_new'] = play_count['pass_count'] + play_count['qb_scramble'] 
play_count['total_plays'] = play_count['pass_count_new'] + play_count['run_count']

play_count['pass_pct'] = play_count['pass_count_new'] / play_count['total_plays'] * 100
play_count['run_pct'] = play_count['run_count'] / play_count['total_plays'] * 100

play_count['pass_pct'] = play_count['pass_pct'].round(2)
play_count['run_pct'] = play_count['run_pct'].round(2)

print(play_count\
    .drop(columns = ['play_type_x',
                     'play_type_y',
                     'qb_scramble',
                     'pass_count'])\
    .query('down == 4'))

    season  down  run_count  pass_count_new  total_plays  pass_pct  run_pct
3     2022   4.0         12            25.0         37.0     67.57    32.43
7     2023   4.0         11            29.0         40.0     72.50    27.50
11    2024   4.0         15            18.0         33.0     54.55    45.45


In [12]:
# Pass_count column is pass attempts + sacks because sacks count as a play
# Rush_count column is 

In [13]:
pbp['play_type'].unique()

array([None, 'kickoff', 'run', 'pass', 'punt', 'no_play', 'field_goal',
       'extra_point', 'qb_kneel', 'qb_spike'], dtype=object)

In [14]:
run_player = lions_run\
    .groupby(['season', 'rusher', 'rusher_id'])\
    .agg({'rushing_yards': ['mean', 'sum'], 
          'rush_attempt': 'count'})

run_player.columns = list(map('_'.join, run_player.columns.values))

run_player.reset_index()

print(run_player.query('season == 2024').sort_values('rush_attempt_count', ascending = False))

                                   rushing_yards_mean  rushing_yards_sum  \
season rusher          rusher_id                                           
2024   J.Gibbs         00-0039139            5.648000             1412.0   
       D.Montgomery    00-0035685            4.189189              775.0   
       C.Reynolds      00-0035567            4.483871              139.0   
       J.Williams      00-0037240            5.545455               61.0   
       J.Goff          00-0033106            0.500000                3.0   
       J.Jefferson     00-0036670            3.666667               22.0   
       S.Vaki          00-0039364            2.333333               14.0   
       K.Raymond       00-0032464            0.000000                0.0   
       St. Brown       00-0036963            3.000000                6.0   
       H.Hooker        00-0038550            3.000000                3.0   
       J.Reeves-Maybin 00-0033557            1.000000                1.0   

           

In [15]:
pbp['qb_kneel'].unique()

array([0., 1.], dtype=float32)

In [16]:
scrambles = league_pbp\
    .query('qb_scramble == 1')

scramble_count = scrambles\
    .groupby(['season'])\
    .agg({'qb_scramble': 'count',
          'yards_gained': 'sum'})

print(scramble_count)


NameError: name 'league_pbp' is not defined

In [17]:
# we need to make a dataframe that has pass count, rush count, qb_scrambles and qb_spikes
# pass count formula = pass_attempts + sacks + qb_scrambles - qb_spikes
# air_yards might be better because you can filter out the stupid plays like qb_spikes

In [18]:
# Create pass count and run count dataframes. Calculate pass vs run percentages
# You can rename a column in the .agg function

league_pbp = pbp.query('season_type == "REG"')

league_pass = pbp.query('play_type == "pass" &\
                          season_type == "REG"')

league_run =  pbp.query('play_type == "run" &\
                          season_type == "REG"')

league_pass_count = league_pass\
    .groupby(['season','posteam','down', 'play_type'])\
    .agg(pass_count = ('play_type', 'count'))

league_pass_count.reset_index(inplace = True)

league_run_count = league_run\
    .groupby(['season', 'posteam','down', 'play_type'])\
    .agg(run_count = ('play_type', 'count'))

league_run_count.reset_index(inplace = True)

league_scrambles = league_pbp\
    .query('qb_scramble == 1')

league_scramble_count = league_scrambles\
    .groupby(['season', 'posteam','down'])\
    .agg({'qb_scramble': 'count'})


# First merge pass_count and run_count
league_play_count = pd.merge(league_pass_count, league_run_count, 
                      on=['season','posteam', 'down'], 
                      how='outer')\
                      .fillna(0)

# Then merge with scramble_count
league_play_count = pd.merge(league_play_count, league_scramble_count,
                      on=['season','posteam','down'], 
                      how='outer')\
                      .fillna(0)


league_play_count['pass_count_new'] = league_play_count['pass_count'] + league_play_count['qb_scramble'] 
league_play_count['total_plays'] = league_play_count['pass_count_new'] + league_play_count['run_count']

league_play_count['pass_pct'] = league_play_count['pass_count_new'] / league_play_count['total_plays'] * 100
league_play_count['run_pct'] = league_play_count['run_count'] / league_play_count['total_plays'] * 100

league_play_count['pass_pct'] = league_play_count['pass_pct'].round(2)
league_play_count['run_pct'] = league_play_count['run_pct'].round(2)

league_play_count['pass_pct_rank'] = league_play_count.groupby(['season','down'])['pass_pct']\
                                                      .rank(method='dense', ascending=False)
league_play_count['run_pct_rank'] = league_play_count.groupby(['season','down'])['run_pct']\
                                                      .rank(method='dense', ascending=False)
print(league_play_count\
    .drop(columns = ['play_type_x',
                     'play_type_y',
                     'qb_scramble',
                     'pass_count'])\
    .query('posteam == "DET"')\
    .to_string(index=False))

 season posteam  down  run_count  pass_count_new  total_plays  pass_pct  run_pct  pass_pct_rank  run_pct_rank
   2022     DET   1.0        244           236.0        480.0     49.17    50.83           15.0          18.0
   2022     DET   2.0        154           205.0        359.0     57.10    42.90           22.0          10.0
   2022     DET   3.0         55           162.0        217.0     74.65    25.35           23.0           9.0
   2022     DET   4.0         12            25.0         37.0     67.57    32.43            8.0          21.0
   2023     DET   1.0        251           240.0        491.0     48.88    51.12           20.0          13.0
   2023     DET   2.0        157           209.0        366.0     57.10    42.90           22.0          10.0
   2023     DET   3.0         60           162.0        222.0     72.97    27.03           24.0           8.0
   2023     DET   4.0         11            29.0         40.0     72.50    27.50            5.0          20.0
   2024   

In [19]:
# FREQUENCY OF FORMATIONS and PERSONNEL GROUPINGS BY DOWN

In [20]:
lions_formation = lions_pbp\
    .query('posteam == "DET" &\
            season_type == "REG" &\
            play_type == "pass" |\
            play_type == "run"'
          )

formation = lions_formation\
    .groupby(['season', 'offense_formation'])\
    .agg({'yards_gained': 'count'})

print(formation)

                          yards_gained
season offense_formation              
2022   EMPTY                        73
       I_FORM                      100
       JUMBO                        15
       PISTOL                       12
       SHOTGUN                     479
       SINGLEBACK                  399
2023   EMPTY                        74
       I_FORM                       62
       JUMBO                         7
       PISTOL                        8
       SHOTGUN                     485
       SINGLEBACK                  343
       WILDCAT                       4


In [34]:
lions_personnel = lions_pbp\
    .query('posteam == "DET" &\
            season_type == "REG" &\
            play_type == "pass" |\
            play_type == "run"'
          )

personnel = lions_personnel\
    .groupby(['season', 'play_type', 'down', 'offense_personnel'])\
    .agg({'yards_gained': 'count'})

print(personnel\
      .query('season == 2022')\
      .to_string(index = True))


                                                           yards_gained
season play_type down offense_personnel                                
2022   pass      1.0  0 RB, 1 TE, 4 WR                                1
                      1 RB, 0 TE, 4 WR                                4
                      1 RB, 1 TE, 3 WR                              141
                      1 RB, 2 TE, 2 WR                               36
                      1 RB, 3 TE, 1 WR                                7
                      2 RB, 1 TE, 2 WR                               23
                      2 RB, 2 TE, 1 WR                                1
                      3 RB, 1 TE, 1 WR                                1
                      6 OL, 1 RB, 0 TE, 3 WR                          2
                      6 OL, 1 RB, 1 TE, 2 WR                          5
                      6 OL, 1 RB, 2 TE, 1 WR                          8
                      6 OL, 1 RB, 3 TE, 0 WR                    